In [2]:
# =====================================================================
# PROJECT: RAKSHA FARM INTELLIGENCE SYSTEM
# COURSE: Fundamentals of AI Using Agriculture Data Set (ANNAM.AI / IIT Ropar)
# DESCRIPTION: Combined Spoilage Risk Predictor & Intent Matching Engine
# =====================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, accuracy_score

# ---------------------------------------------------------------------
# STEP 1: DATA GENERATION & SEEDING (Simulating raksha-data-set & queries)
# ---------------------------------------------------------------------
print("Initializing Data Pipelines...")
np.random.seed(42)
N_SAMPLES = 200

# Base features
crop_types = np.random.choice(['Tomato', 'Onion'], size=N_SAMPLES)
temperatures = np.random.uniform(22, 45, size=N_SAMPLES) # High summer heat spikes
humidity = np.random.uniform(30, 85, size=N_SAMPLES)
storage_available = np.random.choice([0, 1], size=N_SAMPLES, p=[0.6, 0.4]) # Limited cold storage

# Engineering Custom Columns (Engineering domain depth)
transport_delay_hrs = np.random.randint(4, 48, size=N_SAMPLES)
days_post_harvest = np.random.randint(1, 5, size=N_SAMPLES)

# Logical Spoilage Risk Target Allocation Engine
spoilage_risk = []
for i in range(N_SAMPLES):
    score = 0
    if temperatures[i] > 38: score += 3  # Heatwaves accelerate spoilage
    if storage_available[i] == 0: score += 2 # No cold chain structural shield
    if transport_delay_hrs[i] > 24: score += 2
    if crop_types[i] == 'Tomato' and days_post_harvest[i] > 2: score += 2 # Softening tendency

    if score >= 6:
        spoilage_risk.append('High')
    elif score >= 3:
        spoilage_risk.append('Medium')
    else:
        spoilage_risk.append('Low')

df_farm = pd.DataFrame({
    'Crop_Type': crop_types,
    'Field_Temperature_C': temperatures,
    'Relative_Humidity_Pct': humidity,
    'Cold_Storage_Access': storage_available,
    'Transport_Delay_Hours': transport_delay_hrs,
    'Days_Post_Harvest': days_post_harvest,
    'Spoilage_Risk_Level': spoilage_risk
})

print(f"✔ Successfully engineered dataset with shape: {df_farm.shape}")

# Historical Farmer Query Database
historical_queries = [
    "Garmi ki wajah se tamatar kharab ho rahe hain, kya karein?",
    "Pyaaz ko sadne se bachane ke liye cold storage ka taapman kitna rakhein?",
    "Onion price drop in Maharashtra market today?",
    "Tomato crops turning soft and rotting within 2 days of harvest.",
    "How to claim PM Fasal Bima Yojana for heat wave damage?",
    "Subsidies available for setting up local solar cold storage units?",
    "Tomato leaf curl virus mitigation steps.",
    "Mandi rates for high quality Nashik onions."
]
query_intents = ['Pest_Spoilage', 'Storage', 'Market', 'Pest_Spoilage', 'Scheme', 'Storage', 'Pest_Spoilage', 'Market']

# ---------------------------------------------------------------------
# STEP 2: MACHINE LEARNING MODEL (Supervised Spoilage Prediction)
# ---------------------------------------------------------------------
print("\n--- Training Supervised Spoilage Risk Classifier ---")

# Feature Encoding
X = df_farm.drop(columns=['Spoilage_Risk_Level'])
X = pd.get_dummies(X, columns=['Crop_Type'], drop_first=True)
y = df_farm['Spoilage_Risk_Level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f"Model Training Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ---------------------------------------------------------------------
# STEP 3: NATURAL LANGUAGE PROCESSING (Similar Question Retrieval Engine)
# ---------------------------------------------------------------------
print("\n--- Deploying Historical Advisory Knowledge Retriever ---")

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(historical_queries)

def retrieve_similar_queries(new_query, top_n=2):
    new_query_vector = vectorizer.transform([new_query])
    similarities = cosine_similarity(new_query_vector, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[-top_n:][::-1]

    print(f"\nIncoming Live Query: '{new_query}'")
    print("-------------------------------------------------------")
    print("Retrieved Matching Historical Knowledge Tickets:")
    for rank, idx in enumerate(top_indices, 1):
        print(f" Match {rank} [Score: {similarities[idx]:.2f} | Intent: {query_intents[idx]}]: {historical_queries[idx]}")

# Live test scenario mimicking a real-time incoming SMS dashboard alert
live_farmer_alert = "Heat wave warning! My harvested tomatoes are beginning to soften in the field."
retrieve_similar_queries(live_farmer_alert, top_n=2)

Initializing Data Pipelines...
✔ Successfully engineered dataset with shape: (200, 7)

--- Training Supervised Spoilage Risk Classifier ---
Model Training Accuracy: 97.50%

Classification Report:
              precision    recall  f1-score   support

        High       1.00      0.88      0.93         8
         Low       1.00      1.00      1.00        14
      Medium       0.95      1.00      0.97        18

    accuracy                           0.97        40
   macro avg       0.98      0.96      0.97        40
weighted avg       0.98      0.97      0.97        40


--- Deploying Historical Advisory Knowledge Retriever ---

Incoming Live Query: 'Heat wave warning! My harvested tomatoes are beginning to soften in the field.'
-------------------------------------------------------
Retrieved Matching Historical Knowledge Tickets:
 Match 1 [Score: 0.46 | Intent: Scheme]: How to claim PM Fasal Bima Yojana for heat wave damage?
 Match 2 [Score: 0.19 | Intent: Market]: Onion price drop i